# 12 — Beş bölgü ilə təkrar (seed 42–46)

Bütün əsas cədvəllər 5 fərqli qrup əsaslı bölgüdə yenidən hesablanır və **orta ± std** kimi verilir.
Hər seed `s` üçün eyni prosedur (02_v2 → 03 → 04 → 05 → 10), yalnız `random_state = s`:
bölgü, model təsadüfiliyi, obfuskasiya və augmentation seçimləri.

- **Cədvəl 2** — baza modellər (TF-IDF + LR / SVM / RF / NB, uzunluq modeli): F1, recall, FPR.
- **Cədvəl 3** — texnikalar üzrə recall, baza modellər, ssenari B (və A).
- **Cədvəl 4** — LR müdafiələri, ssenari A: baza, normallaşdırma, augmentation (LOTO), hər ikisi (LOTO); insan formatı F1; FPR.
- **Normallaşdırma bütün modellərdə** — min recall (A).
- **Cədvəl 5** — detektorlar: qara siyahı ×2, libinjection, LR ×3: recall, FPR, adi giriş FPR, ən pis texnika (A).
- **HttpParams** — hər seed-in train+val dəsti ilə üst-üstə düşənlər çıxarılmaqla.

MySQL sütunu (Cədvəl 5) burada yoxdur: MySQL icrası yalnız seed 42 test dəstinin payload-ları üçün aparılıb (06, 09, 11).

Seed 42 nəticələri 02–05 notebook-larının nəticələri ilə müqayisə olunur (təkrarlanma yoxlaması).

In [1]:
import sys, time
from pathlib import Path
from urllib.parse import unquote

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import LinearSVC

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import libinj
from blacklist import flags, sanitize_input, sanitize_search
from experiments import (TECH_ORDER, VALID_TECHS, augment, leakage_free, load_clean, load_httpparams,
                         make_split, obfuscate, plain_benign_mask, rates)
from normalize import human_format, normalize

RES = ROOT / "results"
SEEDS = [42, 43, 44, 45, 46]
clean = load_clean()
hp_all = load_httpparams()
print(len(clean), len(hp_all), "libinjection", libinj.VERSION)

30255 30156 libinjection 4.0.1


## Bir seed üçün tam prosedur
TF-IDF (char 1–5, `sublinear_tf`, `min_df=2`) hər təlim dəstində bir dəfə öyrədilir və həmin dəstin bütün grid modelləri üçün paylaşılır; hiperparametr yalnız validation F1-ə görə seçilir.
Gridlər: baza — 02_v2 (LR C ∈ {0.1, 1, 10, 100}, SVM C ∈ {0.01, 0.1, 1, 10}, RF (n, min_leaf) ∈ {(200,1), (200,2), (400,1)}, NB α ∈ {0.001, 0.01, 0.1, 1});
normallaşdırma və augmentation — 04 (LR {1, 10, 100}, SVM {0.1, 1, 10}, RF 200, NB {0.001, 0.01, 0.1}).

In [2]:
def vectorizer():
    return TfidfVectorizer(analyzer="char", ngram_range=(1, 5), lowercase=True,
                           sublinear_tf=True, min_df=2, dtype=np.float32)

def makers(seed):
    return {
        "LR": lambda p: LogisticRegression(C=p, max_iter=2000, random_state=seed),
        "SVM": lambda p: LinearSVC(C=p, random_state=seed),
        "RF": lambda p: (RandomForestClassifier(n_estimators=p[0], min_samples_leaf=p[1], n_jobs=-1, random_state=seed)
                         if isinstance(p, tuple) else RandomForestClassifier(n_estimators=p, n_jobs=-1, random_state=seed)),
        "NB": lambda p: MultinomialNB(alpha=p),
    }

GRID_BASE = {"LR": [0.1, 1, 10, 100], "SVM": [0.01, 0.1, 1, 10], "RF": [(200, 1), (200, 2), (400, 1)],
             "NB": [0.001, 0.01, 0.1, 1.0]}
GRID_DEF = {"LR": [1, 10, 100], "SVM": [0.1, 1, 10], "RF": [200], "NB": [0.001, 0.01, 0.1]}


class Fitted:
    """prep (optional) -> fitted vectorizer -> fitted classifier, predict on raw strings."""
    def __init__(self, vec, clf, prep=None):
        self.vec, self.clf, self.prep = vec, clf, prep
    def predict(self, xs):
        xs = list(xs)
        if self.prep:
            xs = [self.prep(x) for x in xs]
        return self.clf.predict(self.vec.transform(xs)).astype(int)


def fit_tuned(name, grid, seed, X, y, val, prep=None, vec=None):
    xs = [prep(x) for x in X] if prep else list(X)
    vec = vec or vectorizer().fit(xs)
    Xtr = vec.transform(xs)
    vx = vec.transform([prep(x) for x in val.Query] if prep else val.Query)
    best = None
    for p in grid:
        clf = makers(seed)[name](p).fit(Xtr, y)
        f1 = f1_score(val.Label, clf.predict(vx))
        if best is None or f1 > best[2]:
            best = (clf, p, f1)
    return Fitted(vec, best[0], prep), best[1], vec


def length_model(seed, X, y):
    feat = lambda xs: np.log1p(np.asarray([len(q) for q in xs], dtype=float)).reshape(-1, 1)
    m = make_pipeline(FunctionTransformer(feat), StandardScaler(), LogisticRegression(random_state=seed))
    return m.fit(list(X), y)


def vec_det(fn):
    return lambda xs: np.array([fn(x) for x in xs], dtype=int)

BL_INPUT = vec_det(lambda q: flags(q, sanitize_input))
BL_SEARCH = vec_det(lambda q: flags(q, sanitize_search))
LIBINJ = vec_det(libinj.is_sqli)

In [3]:
def obf_recall(predict, obf_df, techniques=TECH_ORDER, scenarios=("A", "B")):
    out = {}
    for sc in scenarios:
        col = f"input_{sc}"
        uniq = pd.unique(obf_df.loc[obf_df.technique.isin(techniques), col])
        pred = dict(zip(uniq, predict(uniq)))
        for t in techniques:
            out[(sc, t)] = obf_df.loc[obf_df.technique == t, col].map(pred).mean()
    return out


def run_seed(s):
    t0 = time.perf_counter()
    rows = []
    add = lambda table, model, metric, value: rows.append(
        {"seed": s, "table": table, "model": model, "metric": metric, "value": float(value)})

    train, val, test = make_split(clean, s)
    y_tr = train.Label.to_numpy()
    mal_test = test.loc[test.Label == 1, "Query"]
    obf_df = obfuscate(mal_test, s)
    test_h = test.assign(Query=test.Query.map(human_format))
    plain = plain_benign_mask(test)
    hp = leakage_free(hp_all, train, val)
    add("meta", "-", "n_train", len(train)); add("meta", "-", "n_test", len(test))
    add("meta", "-", "n_test_malicious", len(mal_test)); add("meta", "-", "n_httpparams", len(hp))

    # Cədvəl 2 + 3: baza modellər
    base, vec = {}, None
    for name in ("LR", "SVM", "RF", "NB"):
        base[name], p, vec = fit_tuned(name, GRID_BASE[name], s, train.Query, y_tr, val, vec=vec)
        r = rates(test.Label, base[name].predict(test.Query))
        for k in ("f1", "recall", "fpr"):
            add("T2_baseline", name, k, r[k])
        add("T2_baseline", name, "f1_human", rates(test_h.Label, base[name].predict(test_h.Query))["f1"])
        for (sc, t), v in obf_recall(base[name].predict, obf_df).items():
            add(f"T3_obf_{sc}", name, t, v)
        add("T3_obf_A", name, "original", base[name].predict(mal_test.map(unquote)).mean())
        add("T3_obf_B", name, "original", base[name].predict(mal_test).mean())
    lm = length_model(s, train.Query, y_tr)
    r = rates(test.Label, lm.predict(list(test.Query)))
    for k in ("f1", "recall", "fpr"):
        add("T2_baseline", "Length-only LR", k, r[k])

    # Normallaşdırma — bütün modellər
    norm, nvec = {}, None
    for name in ("LR", "SVM", "RF", "NB"):
        norm[name], p, nvec = fit_tuned(name, GRID_DEF[name], s, train.Query, y_tr, val, prep=normalize, vec=nvec)
        for cfg, m in (("Baseline", base[name]), ("Normallaşdırma", norm[name])):
            o = obf_recall(m.predict, obf_df, scenarios=("A",))
            rr = rates(test.Label, m.predict(test.Query))
            add("NORM_all", f"{name} | {cfg}", "f1_test", rr["f1"])
            add("NORM_all", f"{name} | {cfg}", "fpr_test", rr["fpr"])
            add("NORM_all", f"{name} | {cfg}", "f1_human", rates(test_h.Label, m.predict(test_h.Query))["f1"])
            add("NORM_all", f"{name} | {cfg}", "min_recall_A", min(o.values()))

    # Cədvəl 4: LR müdafiələri, ssenari A
    def t4(cfg, model, techniques=TECH_ORDER):
        for (sc, t), v in obf_recall(model.predict, obf_df, techniques, ("A",)).items():
            add("T4_defense_A", cfg, t, v)

    def t4_metrics(cfg, model):
        add("T4_defense_A", cfg, "original", model.predict(mal_test.map(unquote)).mean())
        r = rates(test.Label, model.predict(test.Query))
        add("T4_defense_A", cfg, "f1_test", r["f1"]); add("T4_defense_A", cfg, "fpr_test", r["fpr"])
        rh = rates(test_h.Label, model.predict(test_h.Query))
        add("T4_defense_A", cfg, "f1_human", rh["f1"]); add("T4_defense_A", cfg, "fpr_human", rh["fpr"])

    for cfg, m in (("Baza", base["LR"]), ("Normallaşdırma", norm["LR"])):
        t4(cfg, m); t4_metrics(cfg, m)

    aug_full = augment(train, s)
    full, full_p = {}, {}
    for cfg, prep in (("Augmentation", None), ("Hər ikisi", normalize)):
        full[cfg], full_p[cfg], _ = fit_tuned("LR", GRID_DEF["LR"], s, aug_full.Query, aug_full.Label.to_numpy(), val, prep=prep)
        t4_metrics(cfg + " (LOTO)", full[cfg])   # F1/FPR rows: model trained on all techniques (as in 04)
        t4(cfg + " (görmüş)", full[cfg])
    for tech in TECH_ORDER:
        aug_t = augment(train, s, exclude=(tech,))
        for cfg, prep in (("Augmentation", None), ("Hər ikisi", normalize)):
            xs = [prep(x) for x in aug_t.Query] if prep else list(aug_t.Query)
            v = vectorizer().fit(xs)
            clf = makers(s)["LR"](full_p[cfg]).fit(v.transform(xs), aug_t.Label.to_numpy())
            t4(cfg + " (LOTO)", Fitted(v, clf, prep), techniques=[tech])

    # Cədvəl 5: detektorlar
    dets = {"Qara siyahı (sanitizeInput)": BL_INPUT, "Qara siyahı (sanitizeSearch)": BL_SEARCH,
            "libinjection 4.0.1": LIBINJ, "LR baza": base["LR"].predict,
            "LR + normallaşdırma": norm["LR"].predict, "LR + norm. + augmentation": full["Hər ikisi"].predict}
    hp_obf = obfuscate(hp.loc[hp.Label == 1, "Query"], s)
    hp_aw = hp_obf.loc[hp_obf.technique == "alt_whitespace", "input_A"]
    for name, det in dets.items():
        p = det(test.Query)
        r = rates(test.Label, p)
        add("T5_detectors", name, "recall", r["recall"]); add("T5_detectors", name, "fpr", r["fpr"])
        add("T5_detectors", name, "fpr_plain_input", p[plain].mean())
        oa = obf_recall(det, obf_df, VALID_TECHS, ("A",))
        worst = min(oa, key=oa.get)
        add("T5_detectors", name, "worst_recall_A", oa[worst])
        add("T5_detectors", name, f"worst_is::{worst[1]}", 1)
        ph = det(hp.Query)
        rh = rates(hp.Label, ph)
        add("HTTPPARAMS", name, "recall", rh["recall"]); add("HTTPPARAMS", name, "fpr", rh["fpr"])
        add("HTTPPARAMS", name, "recall_alt_whitespace_A", det(hp_aw).mean())
    for name in ("RF", "SVM"):
        rh = rates(hp.Label, base[name].predict(hp.Query))
        add("HTTPPARAMS", f"{name} baza", "recall", rh["recall"]); add("HTTPPARAMS", f"{name} baza", "fpr", rh["fpr"])
    print(f"seed {s}: {time.perf_counter() - t0:.0f} s")
    return rows

In [4]:
all_rows = []
for s in SEEDS:
    all_rows += run_seed(s)
raw = pd.DataFrame(all_rows)
raw.to_csv(RES / "12_multiseed_raw.csv", index=False)
print(raw.shape)

seed 42: 587 s


seed 43: 493 s


seed 44: 574 s


seed 45: 471 s


seed 46: 55004 s


(1445, 5)


## Seed 42 — orijinal notebook-larla uyğunluq
Seed 42 nəticələri 02_v2 / 04 / 05-in saxlanılmış nəticələri ilə eyni olmalıdır.

In [5]:
r42 = raw[raw.seed == 42].set_index(["table", "model", "metric"]).value
b2 = pd.read_csv(RES / "baseline_results_v2.csv").set_index("model")
d4 = pd.read_csv(RES / "04_defense_table_A.csv", index_col=0)
d5 = pd.read_csv(RES / "05_detector_table.csv", index_col=0)
checks = [
    ("LR F1 (02)", b2.loc["Logistic Regression", "f1"], r42[("T2_baseline", "LR", "f1")]),
    ("SVM F1 (02)", b2.loc["Linear SVM", "f1"], r42[("T2_baseline", "SVM", "f1")]),
    ("RF F1 (02)", b2.loc["Random Forest", "f1"], r42[("T2_baseline", "RF", "f1")]),
    ("NB F1 (02)", b2.loc["Multinomial NB", "f1"], r42[("T2_baseline", "NB", "f1")]),
    ("LR alt_whitespace A, baza (04)", d4.loc["alt_whitespace", "Baseline"], r42[("T4_defense_A", "Baza", "alt_whitespace")]),
    ("LR alt_whitespace A, norm (04)", d4.loc["alt_whitespace", "Normallaşdırma"], r42[("T4_defense_A", "Normallaşdırma", "alt_whitespace")]),
    ("LR alt_whitespace A, aug LOTO (04)", d4.loc["alt_whitespace", "Augmentation (LOTO)"], r42[("T4_defense_A", "Augmentation (LOTO)", "alt_whitespace")]),
    ("LR combination A, both LOTO (04)", d4.loc["combination", "Hər ikisi (LOTO)"], r42[("T4_defense_A", "Hər ikisi (LOTO)", "combination")]),
    ("Blacklist input FPR (05)", d5.loc["Blacklist (Input)", "fpr"], r42[("T5_detectors", "Qara siyahı (sanitizeInput)", "fpr")]),
]
chk = pd.DataFrame(checks, columns=["check", "original", "seed42_here"])
chk["abs_diff"] = (chk.original - chk.seed42_here).abs()
chk.to_csv(RES / "12_seed42_reproduction_check.csv", index=False)
chk.round(5)

,check,original,seed42_here,abs_diff
0,LR F1 (02),0.99710,0.99713,0.00003
1,SVM F1 (02),0.99740,0.99742,0.00002
2,RF F1 (02),0.99600,0.99598,0.00002
3,NB F1 (02),0.93630,0.93626,0.00004
4,"LR alt_whitespace A, baza (04)",0.77771,0.77771,0.00000
5,"LR alt_whitespace A, norm (04)",0.99885,0.99885,0.00000
6,"LR alt_whitespace A, aug LOTO (04)",0.82159,0.82217,0.00058
7,"LR combination A, both LOTO (04)",0.99766,0.99766,0.00000
8,Blacklist input FPR (05),0.64853,0.64853,0.00000


## Orta ± std cədvəlləri

In [6]:
def agg(table, scale=100, digits=2):
    d = raw[raw.table == table].groupby(["model", "metric"]).value.agg(["mean", "std", "min", "max", "count"])
    d[["mean", "std", "min", "max"]] *= scale
    return d.round(digits)

def pm(d):
    return d["mean"].map(lambda v: f"{v:.2f}") + " ± " + d["std"].map(lambda v: f"{v:.2f}")

for t in ["T2_baseline", "T3_obf_B", "T3_obf_A", "T4_defense_A", "NORM_all", "T5_detectors", "HTTPPARAMS"]:
    a = agg(t)
    a.to_csv(RES / f"12_{t}_mean_std.csv")

t2 = pm(agg("T2_baseline")).unstack("metric")[["f1", "recall", "fpr", "f1_human"]]
t2.loc[["LR", "SVM", "RF", "NB", "Length-only LR"]]

metric,f1,recall,fpr,f1_human
model,,,,
LR,99.68 ± 0.05,99.43 ± 0.15,0.04 ± 0.04,95.91 ± 1.05
SVM,99.71 ± 0.05,99.49 ± 0.14,0.04 ± 0.04,96.21 ± 0.54
RF,99.57 ± 0.10,99.19 ± 0.23,0.03 ± 0.03,92.77 ± 1.08
NB,93.72 ± 0.17,99.09 ± 0.17,7.48 ± 0.22,93.59 ± 0.14
Length-only LR,64.58 ± 0.98,57.94 ± 1.17,13.00 ± 0.25,NaN


In [7]:
t3 = pm(agg("T3_obf_B")).unstack("model")[["LR", "SVM", "RF", "NB"]]
t3.loc[["original"] + TECH_ORDER]

model,LR,SVM,RF,NB
metric,,,,
original,99.43 ± 0.15,99.49 ± 0.14,99.19 ± 0.23,99.09 ± 0.17
random_case,99.86 ± 0.11,99.88 ± 0.10,99.56 ± 0.18,99.52 ± 0.05
inline_comment_space,99.92 ± 0.10,99.99 ± 0.03,37.88 ± 15.25,99.93 ± 0.08
alt_whitespace,76.25 ± 4.82,75.55 ± 4.79,26.67 ± 5.48,98.20 ± 0.15
alt_whitespace_url,66.61 ± 12.01,54.82 ± 10.94,31.82 ± 11.12,72.90 ± 2.00
url_encode,71.52 ± 28.91,87.67 ± 9.66,4.53 ± 2.98,98.89 ± 0.41
double_url_encode,17.02 ± 14.40,24.80 ± 20.74,0.16 ± 0.11,98.96 ± 0.29
versioned_comment,100.00 ± 0.00,100.00 ± 0.00,95.50 ± 0.33,99.80 ± 0.15
logical_operator,99.98 ± 0.04,100.00 ± 0.00,99.83 ± 0.19,99.81 ± 0.15


In [8]:
t4 = pm(agg("T4_defense_A")).unstack("model")[["Baza", "Normallaşdırma", "Augmentation (LOTO)", "Hər ikisi (LOTO)"]]
t4.loc[["original"] + TECH_ORDER + ["f1_test", "fpr_test", "f1_human", "fpr_human"]]

model,Baza,Normallaşdırma,Augmentation (LOTO),Hər ikisi (LOTO)
metric,,,,
original,99.42 ± 0.17,99.35 ± 0.22,99.35 ± 0.10,99.36 ± 0.12
random_case,99.86 ± 0.11,99.76 ± 0.15,99.80 ± 0.09,99.77 ± 0.11
inline_comment_space,99.92 ± 0.10,99.76 ± 0.11,99.96 ± 0.03,99.80 ± 0.12
alt_whitespace,76.25 ± 4.82,99.76 ± 0.11,86.49 ± 3.39,99.80 ± 0.12
alt_whitespace_url,76.25 ± 4.82,99.76 ± 0.11,99.91 ± 0.03,99.80 ± 0.12
url_encode,99.79 ± 0.11,99.70 ± 0.16,99.71 ± 0.18,99.64 ± 0.21
double_url_encode,71.52 ± 28.91,99.70 ± 0.16,99.85 ± 0.11,99.64 ± 0.21
versioned_comment,100.00 ± 0.00,99.77 ± 0.14,99.96 ± 0.05,99.83 ± 0.11
logical_operator,99.98 ± 0.04,99.91 ± 0.16,99.96 ± 0.08,99.94 ± 0.08


In [9]:
pm(agg("NORM_all")).unstack("metric")

metric,f1_human,f1_test,fpr_test,min_recall_A
model,,,,
LR | Baseline,95.91 ± 1.05,99.68 ± 0.05,0.04 ± 0.04,63.71 ± 21.97
LR | Normallaşdırma,99.63 ± 0.09,99.63 ± 0.09,0.06 ± 0.05,99.56 ± 0.20
NB | Baseline,93.59 ± 0.14,93.72 ± 0.17,7.48 ± 0.22,98.16 ± 0.12
NB | Normallaşdırma,93.55 ± 0.19,93.55 ± 0.19,7.66 ± 0.21,99.28 ± 0.08
RF | Baseline,92.77 ± 1.08,99.57 ± 0.10,0.03 ± 0.03,4.53 ± 2.98
RF | Normallaşdırma,99.58 ± 0.12,99.58 ± 0.12,0.02 ± 0.02,98.96 ± 0.11
SVM | Baseline,96.21 ± 0.54,99.71 ± 0.05,0.04 ± 0.04,75.55 ± 4.79
SVM | Normallaşdırma,99.68 ± 0.09,99.68 ± 0.09,0.04 ± 0.04,99.60 ± 0.15


In [10]:
t5 = pm(agg("T5_detectors")).unstack("metric")[["recall", "fpr", "fpr_plain_input", "worst_recall_A"]]
worst = (raw[(raw.table == "T5_detectors") & raw.metric.str.startswith("worst_is::")]
         .assign(t=lambda d: d.metric.str.replace("worst_is::", "")).groupby("model").t
         .agg(lambda x: ", ".join(f"{k} ({v})" for k, v in x.value_counts().items())))
t5["worst_technique (seed sayı)"] = worst
t5

metric,recall,fpr,fpr_plain_input,worst_recall_A,worst_technique (seed sayı)
model,,,,,
LR + norm. + augmentation,99.36 ± 0.12,0.04 ± 0.03,0.04 ± 0.05,99.67 ± 0.05,"url_encode (3), hex_strings (1), combination (1)"
LR + normallaşdırma,99.35 ± 0.22,0.06 ± 0.05,0.05 ± 0.08,99.67 ± 0.14,"url_encode (2), combination (2), inline_commen..."
LR baza,99.43 ± 0.15,0.04 ± 0.04,0.04 ± 0.08,63.71 ± 21.97,"alt_whitespace (3), double_url_encode (2)"
Qara siyahı (sanitizeInput),97.51 ± 0.46,64.80 ± 0.26,10.72 ± 1.09,91.97 ± 0.48,double_url_encode (5)
Qara siyahı (sanitizeSearch),94.32 ± 0.82,60.63 ± 0.43,1.77 ± 0.21,71.36 ± 1.07,double_url_encode (5)
libinjection 4.0.1,91.49 ± 0.80,37.42 ± 0.76,0.00 ± 0.00,0.00 ± 0.00,double_url_encode (5)


In [11]:
pm(agg("HTTPPARAMS")).unstack("metric")[["recall", "fpr", "recall_alt_whitespace_A"]]

metric,recall,fpr,recall_alt_whitespace_A
model,,,
LR + norm. + augmentation,99.98 ± 0.00,0.02 ± 0.01,100.00 ± 0.00
LR + normallaşdırma,99.98 ± 0.01,0.02 ± 0.01,100.00 ± 0.00
LR baza,96.91 ± 0.75,0.02 ± 0.01,81.83 ± 4.65
Qara siyahı (sanitizeInput),99.49 ± 0.00,8.56 ± 0.09,99.71 ± 0.00
Qara siyahı (sanitizeSearch),96.39 ± 0.01,0.24 ± 0.01,96.59 ± 0.01
RF baza,94.31 ± 0.49,0.01 ± 0.00,NaN
SVM baza,97.23 ± 0.35,0.03 ± 0.02,NaN
libinjection 4.0.1,96.80 ± 0.01,0.00 ± 0.00,97.22 ± 0.02
